# Session 12 · Spark Statistics for Interactive Dashboards

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/02-big-data-systems-with-hadoop/code/session-12-spark-statistics-and-dashboards.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

**Setup:** the Spark examples need PySpark (and Java, which Google Colab already has).

In [1]:
%pip install -q pyspark

### Example 1 · Descriptive statistics at scale

In [2]:
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[2]").appName("stats")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

cities = F.array(*[F.lit(c) for c in ["Karachi", "Lahore", "Islamabad", "Faisalabad"]])
cats = F.array(*[F.lit(c) for c in ["Electronics", "Fashion", "Grocery"]])
orders = (spark.range(50000)
    .withColumn("city", F.element_at(cities, (F.abs(F.hash("id")) % 4 + 1).cast("int")))
    .withColumn("category", F.element_at(cats, (F.abs(F.hash(F.col("id") * 7)) % 3 + 1).cast("int")))
    .withColumn("items", (F.rand(seed=1) * 5 + 1).cast("int"))
    .withColumn("amount", F.round(F.exp(F.randn(seed=2) * 0.6 + 7.5) * F.col("items") / 2, 0))
    .withColumn("month", (F.col("id") % 12 + 1).cast("int")))

orders.select("items", "amount").summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()
q = orders.approxQuantile("amount", [0.5, 0.9, 0.99], 0.001)
print("approx median, p90, p99 of amount:", q)
print("corr(items, amount) =", round(orders.stat.corr("items", "amount"), 3))
orders.agg(F.round(F.skewness("amount"), 2).alias("skewness"),
           F.approx_count_distinct("city").alias("distinct_cities")).show()
orders.stat.crosstab("city", "category").orderBy("city_category").show()

+-------+------------------+------------------+
|summary|             items|            amount|
+-------+------------------+------------------+
|  count|             50000|             50000|
|   mean|           2.99394|        3231.27496|
| stddev|1.4136701332577224|2811.6610773486323|
|    min|                 1|              81.0|
|    25%|                 2|            1342.0|
|    50%|                 3|            2439.0|
|    75%|                 4|            4212.0|
|    max|                 5|           42868.0|
+-------+------------------+------------------+



approx median, p90, p99 of amount: [2438.0, 6561.0, 13353.0]


corr(items, amount) = 0.541


+--------+---------------+
|skewness|distinct_cities|
+--------+---------------+
|    2.55|              4|
+--------+---------------+



+-------------+-----------+-------+-------+
|city_category|Electronics|Fashion|Grocery|
+-------------+-----------+-------+-------+
|   Faisalabad|       4203|   4166|   4147|
|    Islamabad|       4105|   4238|   4106|
|      Karachi|       4155|   4074|   4235|
|       Lahore|       4107|   4266|   4198|
+-------------+-----------+-------+-------+



### Example 2 · KPI tables with groupBy, rollup and pivot

In [3]:
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[2]").appName("kpis")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
cities = F.array(*[F.lit(c) for c in ["Karachi", "Lahore", "Islamabad", "Faisalabad"]])
cats = F.array(*[F.lit(c) for c in ["Electronics", "Fashion", "Grocery"]])
orders = (spark.range(50000)
    .withColumn("city", F.element_at(cities, (F.abs(F.hash("id")) % 4 + 1).cast("int")))
    .withColumn("category", F.element_at(cats, (F.abs(F.hash(F.col("id") * 7)) % 3 + 1).cast("int")))
    .withColumn("items", (F.rand(seed=1) * 5 + 1).cast("int"))
    .withColumn("amount", F.round(F.exp(F.randn(seed=2) * 0.6 + 7.5) * F.col("items") / 2, 0))
    .withColumn("month", (F.col("id") % 12 + 1).cast("int")))

kpi = orders.agg(F.count("*").alias("orders"), F.sum("amount").alias("revenue"),
                 F.round(F.avg("amount"), 0).alias("avg_order")).first()
print(f"KPI cards -> orders: {kpi['orders']:,} | revenue: Rs {kpi['revenue']:,.0f} | avg order: Rs {kpi['avg_order']:,.0f}\n")

print("rollup: revenue by city with grand total (in Rs million)")
(orders.rollup("city").agg(F.round(F.sum("amount") / 1e6, 2).alias("revenue_m"))
       .orderBy(F.col("city").isNull(), "city").show())

print("pivot: revenue (Rs million) by category and quarter")
(orders.withColumn("quarter", F.concat(F.lit("Q"), F.ceil(F.col("month") / 3).cast("string")))
       .groupBy("category").pivot("quarter").agg(F.round(F.sum("amount") / 1e6, 2))
       .orderBy("category").show())

KPI cards -> orders: 50,000 | revenue: Rs 161,563,748 | avg order: Rs 3,231

rollup: revenue by city with grand total (in Rs million)


+----------+---------+
|      city|revenue_m|
+----------+---------+
|Faisalabad|    40.76|
| Islamabad|    39.91|
|   Karachi|    40.17|
|    Lahore|    40.73|
|      NULL|   161.56|
+----------+---------+

pivot: revenue (Rs million) by category and quarter


+-----------+-----+-----+-----+-----+
|   category|   Q1|   Q2|   Q3|   Q4|
+-----------+-----+-----+-----+-----+
|Electronics|13.43|13.63|13.38|13.49|
|    Fashion|13.45|13.41|13.61| 13.2|
|    Grocery| 13.6|13.19|13.55|13.63|
+-----------+-----+-----+-----+-----+



### Example 3 · Trends with window functions

In [4]:
from pyspark.sql import SparkSession, functions as F, Window
spark = (SparkSession.builder.master("local[2]").appName("windows")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

monthly = (spark.range(1, 13).withColumnRenamed("id", "month")
           .withColumn("revenue", F.round(10 + F.col("month") * 0.8 + F.sin(F.col("month")) * 1.5, 2)))  # Rs million
w = Window.orderBy("month")
trend = (monthly
    .withColumn("prev", F.lag("revenue").over(w))
    .withColumn("mom_growth_%", F.round((F.col("revenue") / F.col("prev") - 1) * 100, 1))
    .withColumn("moving_avg_3", F.round(F.avg("revenue").over(w.rowsBetween(-2, 0)), 2))
    .withColumn("running_total", F.round(F.sum("revenue").over(w.rowsBetween(Window.unboundedPreceding, 0)), 2))
    .drop("prev"))
trend.show()

+-----+-------+------------+------------+-------------+
|month|revenue|mom_growth_%|moving_avg_3|running_total|
+-----+-------+------------+------------+-------------+
|    1|  12.06|        NULL|       12.06|        12.06|
|    2|  12.96|         7.5|       12.51|        25.02|
|    3|  12.61|        -2.7|       12.54|        37.63|
|    4|  12.06|        -4.4|       12.54|        49.69|
|    5|  12.56|         4.1|       12.41|        62.25|
|    6|  14.38|        14.5|        13.0|        76.63|
|    7|  16.59|        15.4|       14.51|        93.22|
|    8|  17.88|         7.8|       16.28|        111.1|
|    9|  17.82|        -0.3|       17.43|       128.92|
|   10|  17.18|        -3.6|       17.63|        146.1|
|   11|   17.3|         0.7|       17.43|        163.4|
|   12|   18.8|         8.7|       17.76|        182.2|
+-----+-------+------------+------------+-------------+



### Example 4 · Writing the dashboard's aggregate table

In [5]:
import os, tempfile
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[2]").appName("agg-table")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
cities = F.array(*[F.lit(c) for c in ["Karachi", "Lahore", "Islamabad", "Faisalabad"]])
cats = F.array(*[F.lit(c) for c in ["Electronics", "Fashion", "Grocery"]])
orders = (spark.range(50000)
    .withColumn("city", F.element_at(cities, (F.abs(F.hash("id")) % 4 + 1).cast("int")))
    .withColumn("category", F.element_at(cats, (F.abs(F.hash(F.col("id") * 7)) % 3 + 1).cast("int")))
    .withColumn("amount", F.round(F.exp(F.randn(seed=2) * 0.6 + 7.5), 0))
    .withColumn("month", (F.col("id") % 12 + 1).cast("int")))

agg = (orders.groupBy("month", "city", "category")
       .agg(F.count("*").alias("orders"), F.sum("amount").alias("revenue")))
out = os.path.join(tempfile.mkdtemp(), "sales_agg")    # use out = "sales_agg" to keep it for the dashboard app
agg.write.mode("overwrite").partitionBy("month").parquet(out)

raw_rows, agg_rows = orders.count(), agg.count()
print(f"raw rows: {raw_rows:,} -> dashboard rows: {agg_rows:,} ({raw_rows // agg_rows}x smaller)")
print("month partitions written:", len([d for d in os.listdir(out) if d.startswith("month=")]))
pdf = spark.read.parquet(out).toPandas()            # small enough for pandas / a dashboard
print(pdf.groupby("city")["revenue"].sum().sort_values(ascending=False).round(0).to_string())

raw rows: 50,000 -> dashboard rows: 144 (347x smaller)
month partitions written: 12


city
Lahore        27122087.0
Faisalabad    27088796.0
Islamabad     26884790.0
Karachi       26870679.0
